In [1]:
import json
import os
import re
import time
import requests

from pymilvus import DataType, MilvusClient
from pymilvus import model as milvus_model


# ===================== 配置 =====================
class Config:
    # 本地 Ollama 地址
    OLLAMA_BASE_URL = "http://localhost:11434"
    MILVUS_DB_PATH = "products.db"
    COLLECTION_NAME = "skincare_products"
    # 你本地运行的模型
    MODEL_NAME = "deepseek-r1:1.5b"


# ===================== Milvus 向量数据库封装 =====================
class ProductVectorDB:
    def __init__(self):
        self.embedding_model = milvus_model.DefaultEmbeddingFunction()
        self.embedding_dim = len(self.embedding_model.encode_queries(["test"])[0])

        self.client = MilvusClient(Config.MILVUS_DB_PATH)
        self.collection_name = Config.COLLECTION_NAME
        self._init_collection()

    def _init_collection(self):
        if self.collection_name in self.client.list_collections():
            self.client.drop_collection(self.collection_name)

        schema = self.client.create_schema(
            auto_id=False,
            enable_dynamic_fields=False,
        )
        schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
        schema.add_field(field_name="name", datatype=DataType.VARCHAR, max_length=200)
        schema.add_field(field_name="info", datatype=DataType.VARCHAR, max_length=2000)
        schema.add_field(
            field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=self.embedding_dim
        )

        index_params = self.client.prepare_index_params()
        index_params.add_index(
            field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
        )

        self.client.create_collection(
            collection_name=self.collection_name,
            schema=schema,
            index_params=index_params,
        )

    def encode(self, text: str):
        return self.embedding_model.encode_queries([text])[0]

    def insert_products(self, products):
        data = []
        for p in products:
            data.append(
                {
                    "id": p["id"],
                    "name": p["name"],
                    "info": p["info"],
                    "vector": self.encode(p["name"] + " " + p["info"]),
                }
            )
        self.client.insert(collection_name=self.collection_name, data=data)
        print(f"✅ 已插入 {len(products)} 个产品到 Milvus 向量库")

    def search_product(self, product_name: str):
        res = self.client.search(
            collection_name=self.collection_name,
            data=[self.encode(product_name)],
            filter=f'name like "%{product_name}%"',
            limit=1,
            output_fields=["name", "info"],
        )
        if res and res[0]:
            return res[0][0]["entity"]["info"]
        return "未找到产品信息"


# ===================== 工具类 =====================
class ProductTools:
    def __init__(self, vector_db: ProductVectorDB):
        self.vector_db = vector_db

    def search_web(self, query: str):
        time.sleep(0.5)
        return "小红书美妆趋势：补水保湿、修护屏障、抗老、敏感肌、水光肌、熬夜急救"

    def query_product_database(self, product_name: str):
        print(f"[Tool Call] 从 Milvus 查询产品：{product_name}")
        time.sleep(0.7)
        return self.vector_db.search_product(product_name)

    def generate_emoji(self, context: str):
        time.sleep(0.2)
        if "补水" in context:
            return ["💦", "💧", "✨", "🌊"]
        return ["✨", "🔥", "💖", "💯", "🎉"]


# ===================== AI Agent =====================
class XiaohongshuAgent:
    def __init__(self, tools: ProductTools):
        self.base_url = Config.OLLAMA_BASE_URL
        self.model = Config.MODEL_NAME
        self.tools = tools
        self.available_tools = {
            "search_web": self.tools.search_web,
            "query_product_database": self.tools.query_product_database,
            "generate_emoji": self.tools.generate_emoji,
        }

    def chat(self, messages):
        """调用 Ollama /api/chat 接口"""
        url = f"{self.base_url}/api/chat"
        payload = {
            "model": self.model,
            "messages": messages,
            "stream": False
        }
        resp = requests.post(url, json=payload)
        return resp.json()

    def generate(self, product_name, tone="活泼甜美", max_iter=3):
        messages = [
            {
                "role": "system",
                "content": """你是资深小红书爆款文案专家，生成活泼、真诚、高互动文案。
最终输出严格JSON格式，不要加任何多余解释：
{
  "title": "标题",
  "body": "正文",
  "hashtags": ["标签1","标签2","标签3"],
  "emojis": ["✨","🔥"]
}"""
            },
            {
                "role": "user",
                "content": f"请为产品「{product_name}」生成小红书文案，风格{tone}"
            },
        ]

        for _ in range(max_iter):
            try:
                # 调用本地 Ollama API
                resp = self.chat(messages)
                content = resp["message"]["content"]

                # 解析 JSON
                match = re.search(r"```json\s*(.*?)\s*```", content, re.DOTALL)
                data = json.loads(match.group(1) if match else content)
                return json.dumps(data, ensure_ascii=False, indent=2)

            except Exception as e:
                print(f"⚠️ 生成失败，重试中... {e}")
                continue

        return "生成失败"


# ===================== 10个新产品 =====================
SAMPLE_PRODUCTS = [
    {
        "id": 1,
        "name": "深海蓝藻保湿面膜",
        "info": "蓝藻提取物，深层补水、修护屏障、敏感肌可用",
    },
    {"id": 2, "name": "玻尿酸水光精华", "info": "高浓度玻尿酸，快速补水、提亮肤色"},
    {"id": 3, "name": "烟酰胺美白乳液", "info": "5%烟酰胺，淡化痘印、均匀肤色"},
    {"id": 4, "name": "神经酰胺修护面霜", "info": "修护屏障，舒缓泛红，干敏皮救星"},
    {"id": 5, "name": "茶树控油洁面乳", "info": "深层清洁，控油祛痘，温和不紧绷"},
    {"id": 6, "name": "维A醇抗老精华", "info": "淡化细纹，紧致肌肤，提升弹性"},
    {"id": 7, "name": "金盏花舒缓爽肤水", "info": "舒缓镇静，收缩毛孔，补水保湿"},
    {"id": 8, "name": "维生素C亮肤原液", "info": "抗氧化，提亮肤色，改善暗沉"},
    {"id": 9, "name": "氨基酸泡沫洗面奶", "info": "温和清洁，保湿不紧绷，适合所有肤质"},
    {"id": 10, "name": "胶原蛋白补水面膜", "info": "补充胶原，紧致肌肤，长效保湿"},
]

# ===================== 运行 =====================
if __name__ == "__main__":
    db = ProductVectorDB()
    db.insert_products(SAMPLE_PRODUCTS)

    tools = ProductTools(db)
    agent = XiaohongshuAgent(tools)

    result = agent.generate("深海蓝藻保湿面膜")
    print("\n===== 最终文案 =====")
    print(result)

/home/ericz/miniconda3/envs/deepseek_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/ericz/miniconda3/envs/deepseek_env/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


✅ 已插入 10 个产品到 Milvus 向量库
⚠️ 生成失败，重试中... 'message'
⚠️ 生成失败，重试中... 'message'
⚠️ 生成失败，重试中... 'message'

===== 最终文案 =====
生成失败


I0512 14:20:40.209471   13733 chttp2_transport.cc:1369] unix:/tmp/tmp7afphk0k_products.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0512 14:20:40.209754   13733 chttp2_transport.cc:1401] unix:/tmp/tmp7afphk0k_products.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
